# SLM — Kaggle 2× T4 DDP Training (150M)

**Before running:** Go to `Settings → Accelerator → GPU T4 × 2`.

This notebook trains a **~150M parameter** SLM using **DistributedDataParallel (DDP)** across
both T4 GPUs via `torchrun`. Each GPU holds a full model copy and synchronises gradients via NCCL.

### What this notebook does
1. Verifies both T4 GPUs are available
2. Clones / updates your SLM repo from GitHub
3. Installs dependencies
4. Sets your HuggingFace token (for FineWeb-Edu access)
5. Trains the tokenizer (once, if not already done)
6. Launches `torchrun --nproc_per_node=2 train_kaggle.py`
7. Saves checkpoints to `/kaggle/working/checkpoints_kaggle/`

### DDP at a glance
| | Value |
|---|---|
| GPUs | 2× NVIDIA T4 (16 GB each) |
| Backend | NCCL |
| Per-GPU batch | 8 sequences × 8 accum steps |
| Global eff. batch | **131,072 tokens / optimizer step** |
| Model size | **~152M parameters** |
| VRAM per GPU | ~2.1 GB (14 GB headroom) |
| Expected speed | ~80k tokens/s → ~22h for 50k steps |

In [ ]:
# ── 1. Verify GPU setup ────────────────────────────────────────────────────────
import subprocess, sys, torch

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('⚠  No GPU detected! Go to Settings → Accelerator → GPU T4 × 2')
    sys.exit(1)

gpus = [line.strip() for line in result.stdout.strip().splitlines()]
print(f'Found {len(gpus)} GPU(s):')
for g in gpus:
    print(f'  {g}')

if len(gpus) < 2:
    print('⚠  Only 1 GPU found — DDP requires 2. Change accelerator to "GPU T4 x2".')
    sys.exit(1)

print(f'\nPyTorch {torch.__version__}  |  CUDA {torch.version.cuda}')
print(f'bfloat16 support: {torch.cuda.is_bf16_supported()}')
print(f'NCCL available:   {torch.distributed.is_nccl_available()}')
print('✔  GPU setup OK')

In [ ]:
# ── 2. Clone / update the SLM repo ────────────────────────────────────────────
# Edit GITHUB_REPO to point to your fork.
# For a private repo:  'https://<TOKEN>@github.com/<user>/slm.git'

import os

GITHUB_REPO = 'https://github.com/YOUR_USER/slm.git'   # ← EDIT THIS
PROJECT_DIR = '/kaggle/working/slm'

if os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
    print('Repo already cloned — pulling latest...')
    result = subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print('Cloning repo...')
    result = subprocess.run(['git', 'clone', GITHUB_REPO, PROJECT_DIR],
                            capture_output=True, text=True)
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        print(f'⚠  Clone failed. Check GITHUB_REPO URL and token.')
        sys.exit(1)

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')
print('Files:', [f for f in os.listdir('.') if f.endswith('.py') or f.endswith('.ipynb')])

In [ ]:
# ── 3. Install dependencies ────────────────────────────────────────────────────
# Kaggle already has torch + CUDA. Only the HuggingFace stack is missing.
!pip install -q tokenizers>=0.19.0 datasets>=2.18.0 huggingface-hub>=0.22.0 psutil
print('Dependencies installed.')

In [ ]:
# ── 4. Set HuggingFace token ───────────────────────────────────────────────────
# Option A: Kaggle secret (recommended — add it in Account → Secrets)
# Option B: Paste your token directly (never commit this to GitHub)

import os
from kaggle_secrets import UserSecretsClient  # noqa: only available in Kaggle env

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    if HF_TOKEN:
        os.environ['HF_TOKEN'] = HF_TOKEN
        print('HF_TOKEN loaded from Kaggle secrets.')
    else:
        print('HF_TOKEN secret is empty — add it in Account → Secrets.')
except Exception:
    # Fallback: uncomment the next line and paste your token
    # os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'
    print('Could not load Kaggle secrets — continuing without HF_TOKEN.')
    print('Downloads will be slower (rate-limited). Add HF_TOKEN in Account → Secrets.')

In [ ]:
# ── 5. Show 150M config summary ───────────────────────────────────────────────
import sys
sys.path.insert(0, PROJECT_DIR)

from config_kaggle import model_cfg, train_cfg

d, L, H, Hkv = model_cfg.d_model, model_cfg.n_layers, model_cfg.n_heads, model_cfg.n_kv_heads
B, G, T = train_cfg.batch_size, train_cfg.grad_accum_steps, train_cfg.context_len

print(f'  Architecture : d_model={d}, n_layers={L}, n_heads={H}, kv_heads={Hkv}')
print(f'  d_ff         : {model_cfg.d_ff}')
print(f'  vocab_size   : {model_cfg.vocab_size:,}')
print(f'  context_len  : {T}')
print()
print(f'  Per-GPU batch     : {B} seqs × {G} grad_accum = {B*G} seqs/step')
print(f'  Global eff. batch : {B*G*2} seqs × {T} tokens = {B*G*T*2:,} tokens/step (2 GPUs)')
print(f'  Max steps         : {train_cfg.max_steps:,}')
print(f'  Checkpoint dir    : {train_cfg.checkpoint_dir}')
print(f'  Tokenizer path    : {train_cfg.tokenizer_path}')

In [ ]:
# ── 6. Train tokenizer (runs once; skip if tokenizer_100m/ already exists) ────
# The 150M config reuses the same 32 768-vocab tokenizer as config_100m.
# If you already have tokenizer_100m/ in your repo, this cell is a no-op.

import os
tok_path = os.path.join(PROJECT_DIR, 'tokenizer_100m', 'tokenizer.json')

if os.path.exists(tok_path):
    print(f'Tokenizer found at {tok_path} — skipping training.')
else:
    print('Training tokenizer (vocab=32768, takes ~20-30 min on Kaggle)...')
    ret = subprocess.run(
        [sys.executable, 'tokenizer_train.py', '--config', 'config_kaggle'],
        cwd=PROJECT_DIR
    )
    if ret.returncode == 0:
        print('Tokenizer saved to tokenizer_100m/')
    else:
        print('⚠  Tokenizer training failed. Check output above.')

In [ ]:
# ── 7. (Optional) Resume from a previous checkpoint ───────────────────────────
# Set RESUME_CHECKPOINT to a path if you want to resume from a specific step.
# Leave as None to auto-resume from the latest checkpoint in checkpoints_kaggle/
# (or start from scratch if no checkpoint exists).

RESUME_CHECKPOINT = None   # e.g. 'checkpoints_kaggle/step-0005000'

resume_flag = ''
if RESUME_CHECKPOINT:
    resume_flag = f'--resume {RESUME_CHECKPOINT}'
    print(f'Will resume from: {RESUME_CHECKPOINT}')
else:
    print('Will auto-resume from latest checkpoint (or start fresh).')

In [ ]:
# ── 8. Launch DDP training ─────────────────────────────────────────────────────
#
# torchrun spawns 2 processes (one per GPU) and sets RANK, LOCAL_RANK,
# WORLD_SIZE, MASTER_ADDR, MASTER_PORT automatically.
#
# The --rdzv_backend=c10d flag uses the built-in rendezvous without any
# external coordinator — works perfectly in Kaggle's single-machine environment.
#
# Training output: you will see logs from rank 0 only (rank 1 is silent).
# Checkpoints are saved every 500 steps to checkpoints_kaggle/

import os, subprocess, sys

os.chdir(PROJECT_DIR)

cmd = [
    sys.executable, '-m', 'torch.distributed.run',
    '--nproc_per_node=2',
    '--rdzv_backend=c10d',
    '--rdzv_endpoint=localhost:29500',
    'train_kaggle.py',
]
if RESUME_CHECKPOINT:
    cmd += ['--resume', RESUME_CHECKPOINT]

print('Launching DDP training with command:')
print(' '.join(cmd))
print('─' * 60)

# Run training — this blocks until completion (or interrupt).
# Output streams to this cell in real time.
ret = subprocess.run(cmd, cwd=PROJECT_DIR)

if ret.returncode == 0:
    print('\n✔  Training complete.')
else:
    print(f'\n⚠  Training exited with code {ret.returncode}.')

In [ ]:
# ── 9. List checkpoints ────────────────────────────────────────────────────────
import os
ckpt_dir = os.path.join(PROJECT_DIR, 'checkpoints_kaggle')
if os.path.isdir(ckpt_dir):
    entries = sorted(os.listdir(ckpt_dir))
    print(f'Checkpoints in {ckpt_dir}:')
    for e in entries:
        p = os.path.join(ckpt_dir, e)
        files = os.listdir(p) if os.path.isdir(p) else []
        sizes = [os.path.getsize(os.path.join(p, f)) / 1024**2 for f in files if f.endswith('.pt')]
        total_mb = sum(sizes)
        print(f'  {e}  ({total_mb:.0f} MB)')
else:
    print('No checkpoints found yet.')

In [ ]:
# ── 10. Quick inference check ──────────────────────────────────────────────────
# Load the latest checkpoint and run a few generation samples.

import sys, os, torch
sys.path.insert(0, PROJECT_DIR)

from config_kaggle import model_cfg, train_cfg
from model import SLM
from checkpoint import load_checkpoint
from inference import run_inference_check
from tokenizers import Tokenizer

# Load tokenizer
tok_file = os.path.join(PROJECT_DIR, train_cfg.tokenizer_path, 'tokenizer.json')
if not os.path.exists(tok_file):
    print('Tokenizer not found — run cell 6 first.')
else:
    tokenizer = Tokenizer.from_file(tok_file)

    # Load model
    device = torch.device('cuda:0')
    model  = SLM(model_cfg).to(device=device, dtype=torch.bfloat16)

    # Dummy optimisers just for load_checkpoint signature
    from optimizer import build_optimizers
    muon_opt, adamw_opt = build_optimizers(model, train_cfg)

    step, tokens = load_checkpoint(model, muon_opt, adamw_opt, train_cfg)
    model.eval()
    print(f'Loaded checkpoint at step {step:,}  ({tokens/1e6:.1f}M tokens seen)')
    print()

    prompts = [
        'The theory of relativity states that',
        'Once upon a time in a kingdom far away',
        'The best way to learn mathematics is',
    ]
    results = run_inference_check(
        model, tokenizer, prompts=prompts,
        max_new_tokens=100, temperature=0.8, top_k=40,
    )
    for prompt, gen in zip(prompts, results):
        print(f'PROMPT : {prompt}')
        print(f'OUTPUT : {gen}')
        print()

In [ ]:
# ── 11. Plot training loss from JSONL log ─────────────────────────────────────
import json, os
import matplotlib.pyplot as plt

log_dir = os.path.join(PROJECT_DIR, 'logs_kaggle')
records = []
if os.path.isdir(log_dir):
    for fname in sorted(os.listdir(log_dir)):
        if fname.endswith('.jsonl'):
            with open(os.path.join(log_dir, fname)) as f:
                for line in f:
                    try:
                        records.append(json.loads(line))
                    except json.JSONDecodeError:
                        pass

if not records:
    print('No log records found yet.')
else:
    steps      = [r['step']     for r in records if 'loss' in r]
    train_loss = [r['loss']     for r in records if 'loss' in r]
    val_steps  = [r['step']     for r in records if r.get('val_loss') is not None]
    val_loss   = [r['val_loss'] for r in records if r.get('val_loss') is not None]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(steps, train_loss, lw=0.8, alpha=0.7, label='Train loss')
    if val_steps:
        ax.plot(val_steps, val_loss, lw=1.5, marker='o', ms=3, label='Val loss')
    ax.set_xlabel('Step')
    ax.set_ylabel('Cross-entropy loss')
    ax.set_title('SLM 150M — Kaggle 2× T4 DDP Training')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_DIR, 'training_loss.png'), dpi=150)
    plt.show()
    print(f'Last train loss: {train_loss[-1]:.4f} at step {steps[-1]:,}')

## Tips & Troubleshooting

| Issue | Fix |
|---|---|
| Only 1 GPU detected | Settings → Accelerator → **GPU T4 × 2** |
| `NCCL error` on launch | Re-run cell 8; port 29500 may be in use — change `rdzv_endpoint` to `:29501` |
| CUDA OOM | Unlikely (2.1 GB/16 GB), but reduce `batch_size` to 4 in `config_kaggle.py` |
| `ModuleNotFoundError` | Run `os.chdir(PROJECT_DIR)` and re-run the failing cell |
| Slow downloads | Add `HF_TOKEN` in Kaggle Account → Secrets → Add New Secret |
| Interrupt mid-training | Ctrl+C inside cell 8 saves an emergency checkpoint. Re-run cell 8 to resume. |
| Session timeout | Kaggle notebooks can run up to 12h. For 50k steps (~22h), split into two sessions using `--resume`. |

### How DDP works here

```
torchrun spawns 2 processes:

  Process 0 (rank=0, LOCAL_RANK=0, GPU cuda:0)
    - forward + backward on its own data shard
    - all-reduce gradients with rank 1  (NCCL)
    - optimizer.step()                  (Muon + AdamW)
    - writes checkpoints + logs         (rank 0 only)

  Process 1 (rank=1, LOCAL_RANK=1, GPU cuda:1)
    - forward + backward on its own data shard
    - all-reduce gradients with rank 0  (NCCL)
    - optimizer.step()                  (must call even though rank 0 logs)
    - silent (no logging/checkpointing)

  Both processes are synchronised via dist.barrier() after each optimizer step.
```

### Effective batch size calculation

```
2 GPUs × batch_size(8) × grad_accum(8) × context_len(1024)
= 131,072 tokens per optimizer step
```

This is well above the ~65k tokens/step used in the 100M CPU config, meaning cleaner gradients
and faster loss convergence per step.

### Model architecture (150M)

| Hyperparameter | Value |
|---|---|
| Parameters | **~152 M** |
| Vocabulary | 32 768 tokens (BPE) |
| Context length | 1 024 tokens |
| `d_model` | 960 |
| Layers | 16 |
| Q heads | 16 |
| KV heads (GQA) | 4 |
| `d_ff` | 2 560 |
| Optimizer | Muon (matrix) + AdamW (embed/norm) |
| Dtype | bfloat16 |